# Day 5 — Book Recommender ML Pipeline
## Notebook: `02_books_pipeline.ipynb`

**Goal:** Build the complete sklearn Pipeline that transforms raw Book-Crossing CSVs  
into 4 trained model pickles ready for the FastAPI backend.

**Pipeline stages:**  
`DataLoader` → `BookCleaner` → `UserCleaner` → `DataMerger` → `RatingFilter`  
→ `MatrixBuilder` → `KNNRecommender` → `PopularityBuilder`

**Output pickles** (saved to `../models/`):  
- `book_model.pkl` — fitted KNN model  
- `book_pivot.pkl` — user-book rating matrix  
- `popular_df.pkl` — popularity table with Bayesian scores + cover URLs  
- `books_meta.pkl` — Book-Title → Author + Image-URL-M  

> Run all cells top to bottom. CSVs must be in `../data/`

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

os.makedirs('../models', exist_ok=True)
print("Imports successful — pipeline ready to build")

Imports successful — pipeline ready to build


## 1. Custom transformer: DataLoader

In [2]:
class DataLoader(BaseEstimator, TransformerMixin):
    """
    Stage 1: Load all three raw CSVs.
    Also saves books_raw (with image URLs intact) BEFORE BookCleaner
    strips image columns — this is the fix that ensures cover images
    are available in books_meta.pkl for the FastAPI backend.
    
    Input  : None (pipeline starts here)
    Output : dict with keys books, books_raw, ratings, users
    """
    def __init__(self,
                 books_path='../data/Books.csv',
                 ratings_path='../data/Ratings.csv',
                 users_path='../data/Users.csv'):
        self.books_path   = books_path
        self.ratings_path = ratings_path
        self.users_path   = users_path

    def fit(self, X=None, y=None):
        return self

    def transform(self, X=None):
        books   = pd.read_csv(self.books_path,   low_memory=False)
        ratings = pd.read_csv(self.ratings_path, low_memory=False)
        users   = pd.read_csv(self.users_path,   low_memory=False)

        # Save slim metadata WITH image URLs before BookCleaner drops them
        books_raw = books[['Book-Title', 'Book-Author', 'Image-URL-M']].copy()
        books_raw = books_raw.drop_duplicates('Book-Title')

        print(f"[DataLoader]   books={books.shape}  ratings={ratings.shape}  users={users.shape}")
        return {
            'books'    : books,
            'books_raw': books_raw,   # keeps Image-URL-M for cover lookups
            'ratings'  : ratings,
            'users'    : users,
        }

print("DataLoader defined")

DataLoader defined


## 2. Custom transformer: BookCleaner

In [3]:
class BookCleaner(BaseEstimator, TransformerMixin):
    """
    Stage 2: Clean the books DataFrame.
    - Drop all Image-URL columns (not needed for ML; covers come from books_raw)
    - Drop rows missing Book-Title, Book-Author, or Publisher
    - Coerce Year-Of-Publication to numeric and filter 1900-2025
    
    Input / Output : data dict (mutates 'books' key)
    """
    def __init__(self, year_min=1900, year_max=2025):
        self.year_min = year_min
        self.year_max = year_max

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['books'].copy()

        # Drop image URL columns
        img_cols = [c for c in df.columns if 'Image-URL' in c]
        df.drop(columns=img_cols, inplace=True)
        
        # Drop rows missing critical fields
        df.dropna(subset=['Book-Title', 'Book-Author', 'Publisher'], inplace=True)

        # Fix Year-Of-Publication
        df['Year-Of-Publication'] = pd.to_numeric(
            df['Year-Of-Publication'], errors='coerce')
        df = df[
            (df['Year-Of-Publication'] >= self.year_min) &
            (df['Year-Of-Publication'] <= self.year_max)
        ]

        df.reset_index(drop=True, inplace=True)
        data['books'] = df
        print(f"[BookCleaner]  books after cleaning: {df.shape}")
        return data

print("BookCleaner defined")

BookCleaner defined


## 3. Custom transformer: UserCleaner

In [4]:
class UserCleaner(BaseEstimator, TransformerMixin):
    """
    Stage 3: Clean the users DataFrame.
    Age is 39% missing and not used by the CF model — keep only User-ID.
    
    Input / Output : data dict (mutates 'users' key)
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        data['users'] = data['users'][['User-ID']].copy()
        print(f"[UserCleaner]  users after cleaning: {data['users'].shape}")
        return data

print("UserCleaner defined")

UserCleaner defined


## 4. Custom transformer: DataMerger

In [5]:
class DataMerger(BaseEstimator, TransformerMixin):
    """
    Stage 4: Merge books + ratings + users into one flat DataFrame.
    Uses books_raw (saved by DataLoader) as books_meta so cover
    images are preserved even after BookCleaner drops image columns.
    
    Input  : data dict
    Output : data dict (adds 'book_ratings' and 'books_meta' keys)
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data    = X.copy()
        books   = data['books']
        ratings = data['ratings']
        users   = data['users']

        # books_meta comes from books_raw — has Image-URL-M intact
        data['books_meta'] = data['books_raw'].copy()

        merged = ratings.merge(books, on='ISBN', how='inner')
        merged = merged.merge(users, on='User-ID', how='inner')
        merged = merged[['ISBN','Book-Title','Book-Author','User-ID','Book-Rating']]

        data['book_ratings'] = merged
        print(f"[DataMerger]   merged shape: {merged.shape}")
        return data

print("DataMerger defined")

DataMerger defined


## 5. Custom transformer: RatingFilter

In [6]:
class RatingFilter(BaseEstimator, TransformerMixin):
    """
    Stage 5: Remove noise to improve collaborative filtering quality.
    
    Steps:
    1. Remove implicit ratings (Book-Rating == 0)
    2. Keep only users who rated >= min_user_ratings books
    3. Keep only books with >= min_book_ratings ratings  
    4. Deduplicate: same (User-ID, Book-Title) → keep max rating
       (handles same book available under multiple ISBNs)
    
    Input / Output : data dict (adds 'filtered' key)
    """
    def __init__(self, min_user_ratings=50, min_book_ratings=50):
        self.min_user_ratings = min_user_ratings
        self.min_book_ratings = min_book_ratings

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['book_ratings'].copy()

        # Step 1: Remove implicit (zero) ratings
        df = df[df['Book-Rating'] > 0]
        print(f"[RatingFilter] after implicit removal   : {len(df):,} ratings")

        # Step 2: Active users
        active = df['User-ID'].value_counts()
        active = active[active >= self.min_user_ratings].index
        df = df[df['User-ID'].isin(active)]
        print(f"[RatingFilter] after active-user filter : {len(df):,} ratings  "
              f"({df['User-ID'].nunique()} users)")

        # Step 3: Popular books
        popular = df['Book-Title'].value_counts()
        popular = popular[popular >= self.min_book_ratings].index
        df = df[df['Book-Title'].isin(popular)]
        print(f"[RatingFilter] after popular-book filter: {len(df):,} ratings  "
              f"({df['Book-Title'].nunique()} books)")

        # Step 4: Deduplicate
        df = (df.groupby(['User-ID', 'Book-Title'])['Book-Rating']
                .max()
                .reset_index())
        print(f"[RatingFilter] after deduplication      : {len(df):,} entries")

        data['filtered'] = df
        return data

print("RatingFilter defined")

RatingFilter defined


## 6. Custom transformer: MatrixBuilder

In [7]:
class MatrixBuilder(BaseEstimator, TransformerMixin):
    """
    Stage 6: Build the User-Book pivot matrix and convert to CSR sparse format.
    
    Pivot layout (item-based collaborative filtering):
      Rows    = Book-Title  (we find books similar to other books)
      Columns = User-ID     (users are features)
      Values  = Book-Rating (0 where unrated after fillna)
    
    Input / Output : data dict (adds 'pivot_table' and 'book_sparse' keys)
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['filtered']

        pivot = df.pivot_table(
            index='Book-Title',
            columns='User-ID',
            values='Book-Rating'
        ).fillna(0)

        sparse  = csr_matrix(pivot)
        density = sparse.nnz / (sparse.shape[0] * sparse.shape[1])

        data['pivot_table'] = pivot
        data['book_sparse'] = sparse
        print(f"[MatrixBuilder] pivot table : {pivot.shape}")
        print(f"[MatrixBuilder] sparsity    : {1-density:.4%}")
        return data

print("MatrixBuilder defined")

MatrixBuilder defined


## 7. Custom transformer: KNNRecommender

In [8]:
class KNNRecommender(BaseEstimator, TransformerMixin):
    """
    Stage 7: Fit a KNN model (cosine similarity) on the sparse user-book matrix.
    
    n_neighbors=11 supports returning up to 10 results
    (index 0 is always the query book itself — we skip it).
    
    Input  : data dict
    Output : data dict (adds 'knn_model' key)
    """
    def __init__(self, n_neighbors=11, metric='cosine', algorithm='brute'):
        self.n_neighbors = n_neighbors
        self.metric      = metric
        self.algorithm   = algorithm

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data   = X.copy()
        sparse = data['book_sparse']

        knn = NearestNeighbors(
            metric     = self.metric,
            algorithm  = self.algorithm,
            n_neighbors= self.n_neighbors,
            n_jobs     = -1
        )
        knn.fit(sparse)
        data['knn_model'] = knn
        print(f"[KNNRecommender] fitted on {sparse.shape[0]} books  "
              f"x  {sparse.shape[1]} users")
        return data

print("KNNRecommender defined")

KNNRecommender defined


## 8. Custom transformer: PopularityBuilder

In [9]:
class PopularityBuilder(BaseEstimator, TransformerMixin):
    """
    Stage 8: Build a popularity table for the homepage / cold-start problem.
    
    Uses Bayesian average to balance volume with quality:
      score = (v * R + m * C) / (v + m)
    
    where:
      v = number of ratings for this book
      R = average rating for this book
      m = minimum ratings threshold (50)
      C = global mean rating across all books
    
    Attaches cover image URLs from books_raw (Image-URL-M).
    Input / Output : data dict (adds 'popularity_df' key)
    """
    def __init__(self, min_ratings=50):
        self.min_ratings = min_ratings

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        br   = data['book_ratings']

        pop = (br[br['Book-Rating'] > 0]
               .groupby('Book-Title')
               .agg(
                   num_ratings = ('Book-Rating', 'count'),
                   avg_rating  = ('Book-Rating', 'mean')
               )
               .reset_index())

        m = self.min_ratings
        C = pop['avg_rating'].mean()
        pop = pop[pop['num_ratings'] >= m].copy()
        v = pop['num_ratings']
        R = pop['avg_rating']
        pop['score'] = (v * R + m * C) / (v + m)
        pop = pop.sort_values('score', ascending=False).reset_index(drop=True)

        # Attach cover URLs and author from books_raw
        covers = data['books_raw'][['Book-Title','Book-Author','Image-URL-M']].copy()
        covers = covers.drop_duplicates('Book-Title')
        pop = pop.merge(covers, on='Book-Title', how='left')

        data['popularity_df'] = pop
        print(f"[PopularityBuilder] popularity table: {pop.shape}")
        print(f"[PopularityBuilder] top book: {pop.iloc[0]['Book-Title']}")
        return data

print("PopularityBuilder defined")

PopularityBuilder defined


## 9. Assemble and run the full pipeline

In [10]:
pipeline = Pipeline(steps=[
    ('loader',      DataLoader()),
    ('book_clean',  BookCleaner(year_min=1900, year_max=2025)),
    ('user_clean',  UserCleaner()),
    ('merger',      DataMerger()),
    ('filter',      RatingFilter(min_user_ratings=50, min_book_ratings=50)),
    ('matrix',      MatrixBuilder()),
    ('knn',         KNNRecommender(n_neighbors=11, metric='cosine')),
    ('popularity',  PopularityBuilder(min_ratings=50)),
])

print("Pipeline stages:")
for name, step in pipeline.steps:
    print(f"  {name:15s} → {step.__class__.__name__}")

Pipeline stages:
  loader          → DataLoader
  book_clean      → BookCleaner
  user_clean      → UserCleaner
  merger          → DataMerger
  filter          → RatingFilter
  matrix          → MatrixBuilder
  knn             → KNNRecommender
  popularity      → PopularityBuilder


In [11]:
# Run the full pipeline
print("Running pipeline...\n")
output = pipeline.fit_transform(None)

print("\nAvailable output keys:", list(output.keys()))

Running pipeline...

[DataLoader]   books=(271360, 8)  ratings=(1149780, 3)  users=(278858, 3)
[BookCleaner]  books after cleaning: (266719, 5)
[UserCleaner]  users after cleaning: (278858, 1)
[DataMerger]   merged shape: (1017062, 5)
[RatingFilter] after implicit removal   : 378,027 ratings
[RatingFilter] after active-user filter : 150,966 ratings  (1163 users)
[RatingFilter] after popular-book filter: 4,338 ratings  (64 books)
[RatingFilter] after deduplication      : 4,248 entries
[MatrixBuilder] pivot table : (64, 948)
[MatrixBuilder] sparsity    : 92.9984%
[KNNRecommender] fitted on 64 books  x  948 users
[PopularityBuilder] popularity table: (645, 6)
[PopularityBuilder] top book: Harry Potter and the Goblet of Fire (Book 4)

Available output keys: ['books', 'books_raw', 'ratings', 'users', 'books_meta', 'book_ratings', 'filtered', 'pivot_table', 'book_sparse', 'knn_model', 'popularity_df']


## 10. Recommend function — test before pickling

In [12]:
def recommend_books(book_name: str, n: int = 10,
                    pivot=None, sparse=None, knn=None):
    """
    Returns up to n similar book recommendations.
    
    Parameters
    ----------
    book_name : str  — exact title as it appears in the dataset
    n         : int  — number of results (default 10, max 10)
    pivot     : DataFrame — defaults to pipeline output
    sparse    : csr_matrix — defaults to pipeline output
    knn       : NearestNeighbors — defaults to pipeline output
    
    Returns
    -------
    list of (title, cosine_distance) tuples
    """
    if pivot is None:  pivot  = output['pivot_table']
    if sparse is None: sparse = output['book_sparse']
    if knn is None:    knn    = output['knn_model']

    if book_name not in pivot.index:
        matches = [t for t in pivot.index if book_name.lower() in t.lower()][:5]
        print(f'"{book_name}" not found. Close matches:')
        for m in matches:
            print(f'  - {m}')
        return []

    idx = pivot.index.get_loc(book_name)
    vec = sparse[idx]
    distances, indices = knn.kneighbors(vec, n_neighbors=n + 1)

    results = []
    for dist, i in zip(distances[0][1:], indices[0][1:]):
        results.append((pivot.index[i], round(float(dist), 4)))
    return results

# Smoke test
sample = output['pivot_table'].index[0]
print(f'Recommendations for: "{sample}"\n')
for rank, (title, dist) in enumerate(recommend_books(sample), 1):
    sim = round((1 - dist) * 100, 1)
    print(f'  {rank:2}. [{sim:5.1f}% match]  {title}')

Recommendations for: "1st to Die: A Novel"

   1. [ 26.9% match]  The Summons
   2. [ 25.4% match]  2nd Chance
   3. [ 18.2% match]  The Partner
   4. [ 18.1% match]  The Client
   5. [ 17.2% match]  Dreamcatcher
   6. [ 16.9% match]  Snow Falling on Cedars
   7. [ 16.2% match]  Summer Sisters
   8. [ 16.0% match]  Hannibal
   9. [ 15.9% match]  The Secret Life of Bees
  10. [ 15.7% match]  Angels &amp; Demons


## 11. Verify books_meta has cover images

In [13]:
meta = output['books_meta']
print(f"books_meta shape: {meta.shape}")
print(f"Columns: {meta.columns.tolist()}")
print(f"Has Image-URL-M: {'Image-URL-M' in meta.columns}")
print()

# Check a few cover URLs
sample_covers = meta[meta['Image-URL-M'].notna()].head(3)
for _, row in sample_covers.iterrows():
    print(f"Title : {row['Book-Title'][:50]}")
    print(f"Author: {row['Book-Author']}")
    print(f"Cover : {row['Image-URL-M']}")
    print()

books_meta shape: (242135, 3)
Columns: ['Book-Title', 'Book-Author', 'Image-URL-M']
Has Image-URL-M: True

Title : Classical Mythology
Author: Mark P. O. Morford
Cover : http://images.amazon.com/images/P/0195153448.01.MZZZZZZZ.jpg

Title : Clara Callan
Author: Richard Bruce Wright
Cover : http://images.amazon.com/images/P/0002005018.01.MZZZZZZZ.jpg

Title : Decision in Normandy
Author: Carlo D'Este
Cover : http://images.amazon.com/images/P/0060973129.01.MZZZZZZZ.jpg



## 12. Export all 4 pickles to models/

In [14]:
artifacts = {
    '../models/book_model.pkl'  : output['knn_model'],
    '../models/book_pivot.pkl'  : output['pivot_table'],
    '../models/popular_df.pkl'  : output['popularity_df'],
    '../models/books_meta.pkl'  : output['books_meta'],
}

for path, obj in artifacts.items():
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  Saved: {path:40s}  ({size_mb:.1f} MB)")

print("\nAll 4 book pickles saved to ../models/")

  Saved: ../models/book_model.pkl                  (0.0 MB)
  Saved: ../models/book_pivot.pkl                  (0.5 MB)
  Saved: ../models/popular_df.pkl                  (0.1 MB)
  Saved: ../models/books_meta.pkl                  (28.6 MB)

All 4 book pickles saved to ../models/


## 13. Verify pickles round-trip correctly

In [15]:
# Load every pickle and confirm it works
with open('../models/book_model.pkl',  'rb') as f: chk_model  = pickle.load(f)
with open('../models/book_pivot.pkl',  'rb') as f: chk_pivot  = pickle.load(f)
with open('../models/popular_df.pkl',  'rb') as f: chk_pop    = pickle.load(f)
with open('../models/books_meta.pkl',  'rb') as f: chk_meta   = pickle.load(f)

print("Pickle verification:")
print(f"  book_model.pkl  : {type(chk_model).__name__}  n_neighbors={chk_model.n_neighbors}")
print(f"  book_pivot.pkl  : DataFrame {chk_pivot.shape}")
print(f"  popular_df.pkl  : DataFrame {chk_pop.shape}  top score={chk_pop['score'].max():.3f}")
print(f"  books_meta.pkl  : DataFrame {chk_meta.shape}  has covers={'Image-URL-M' in chk_meta.columns}")

# End-to-end test from loaded pickles
test_title = chk_pivot.index[5]
print(f"\nEnd-to-end test from pickles for: '{test_title}'")
results = recommend_books(test_title, pivot=chk_pivot,
                          sparse=csr_matrix(chk_pivot), knn=chk_model)
for rank, (title, dist) in enumerate(results[:5], 1):
    print(f"  {rank}. {title}  ({round((1-dist)*100,1)}% match)")

print("\nAll checks passed — Day 5 complete!")

Pickle verification:
  book_model.pkl  : NearestNeighbors  n_neighbors=11
  book_pivot.pkl  : DataFrame (64, 948)
  popular_df.pkl  : DataFrame (645, 6)  top score=8.856
  books_meta.pkl  : DataFrame (242135, 3)  has covers=True

End-to-end test from pickles for: 'Bridget Jones's Diary'
  1. Harry Potter and the Goblet of Fire (Book 4)  (21.0% match)
  2. Harry Potter and the Chamber of Secrets (Book 2)  (20.7% match)
  3. Message in a Bottle  (19.5% match)
  4. Good in Bed  (18.1% match)
  5. The Red Tent (Bestselling Backlist)  (18.0% match)

All checks passed — Day 5 complete!
